# DA 16주차 - 검증하고 예측하기 (07-1 통계적으로 추론하기)
Colab에서 위에서부터 순서대로 '모두 실행' 하면 됩니다. 각 셀(코드+출력)을 스크린샷해서 노션에 붙여넣으세요.

## 표준점수(z 점수) 구하기

In [ ]:
import numpy as np

x = [0, 3, 5, 7, 10]

s = np.std(x)
m = np.mean(x)
z = (7 - m) / s
print(z)

In [ ]:
from scipy import stats

stats.zscore(x)

## 누적분포 이해하기

In [ ]:
stats.norm.cdf(0)

In [ ]:
stats.norm.cdf(1.0) - stats.norm.cdf(-1.0)

In [ ]:
stats.norm.cdf(2.0) - stats.norm.cdf(-2.0)

In [ ]:
stats.norm.ppf(0.9)

## 중심극한정리 알아보기

In [ ]:
import gdown
gdown.download('https://bit.ly/3pK7iuu', 'ns_book7.csv', quiet=False)

import pandas as pd
ns_book7 = pd.read_csv('ns_book7.csv', low_memory=False)
ns_book7.head()

In [ ]:
import matplotlib.pyplot as plt

plt.hist(ns_book7['대출건수'], bins=50)
plt.yscale('log')
plt.show()

## 샘플링하기

In [ ]:
np.random.seed(42)
sample_means = []
for _ in range(1000):
    m = ns_book7['대출건수'].sample(30).mean()
    sample_means.append(m)

In [ ]:
plt.hist(sample_means, bins=30)
plt.show()

## 샘플링 크기와 정확도

In [ ]:
np.mean(sample_means)

In [ ]:
ns_book7['대출건수'].mean()

In [ ]:
np.random.seed(42)
sample_means = []
for _ in range(1000):
    m = ns_book7['대출건수'].sample(20).mean()
    sample_means.append(m)
np.mean(sample_means)

In [ ]:
np.random.seed(42)
sample_means = []
for _ in range(1000):
    m = ns_book7['대출건수'].sample(40).mean()
    sample_means.append(m)
np.mean(sample_means)

In [ ]:
np.std(sample_means)

In [ ]:
np.std(ns_book7['대출건수']) / np.sqrt(40)

## 모집단의 평균 범위 추정하기: 신뢰구간

In [ ]:
python_books_index = ns_book7['주제분류번호'].str.startswith('00') & \
                     ns_book7['도서명'].str.contains('파이썬')
python_books = ns_book7[python_books_index]
python_books.head()

In [ ]:
len(python_books)

In [ ]:
python_mean = np.mean(python_books['대출건수'])
python_mean

In [ ]:
python_std = np.std(python_books['대출건수'])
python_se = python_std / np.sqrt(len(python_books))
python_se

In [ ]:
stats.norm.ppf(0.975)

In [ ]:
stats.norm.ppf(0.025)

In [ ]:
print(python_mean - 1.96 * python_se, python_mean + 1.96 * python_se)

## 통계적 의미 확인하기: 가설검정 (z 점수로 가설 검증하기)

In [ ]:
cplus_books_index = ns_book7['주제분류번호'].str.startswith('00') & \
                    ns_book7['도서명'].str.contains('C++', regex=False)
cplus_books = ns_book7[cplus_books_index]
cplus_books.head()

In [ ]:
len(cplus_books)

In [ ]:
cplus_mean = np.mean(cplus_books['대출건수'])
cplus_mean

In [ ]:
cplus_se = np.std(cplus_books['대출건수']) / np.sqrt(len(cplus_books))
cplus_se

In [ ]:
(python_mean - cplus_mean) / np.sqrt(python_se**2 + cplus_se**2)

In [ ]:
stats.norm.cdf(2.50)

## t-검정으로 가설 검증하기

In [ ]:
t, pvalue = stats.ttest_ind(python_books['대출건수'], cplus_books['대출건수'])
print(t, pvalue)

## 정규분포가 아닐 때 가설 검증하기: 순열검정

In [ ]:
def statistic(x, y):
    return np.mean(x) - np.mean(y)

In [ ]:
def permutation_test(x, y):
    # 표본의 평균 차이를 계산합니다.
    obs_diff = statistic(x, y)
    # 두 표본을 합칩니다.
    all = np.append(x, y)
    diffs = []
    np.random.seed(42)
    # 순열 검정을 1000번 반복합니다.
    for _ in range(1000):
        # 전체 인덱스를 섞습니다.
        idx = np.random.permutation(len(all))
        # 랜덤하게 두 그룹으로 나눈 다음 평균 차이를 계산합니다.
        x_ = all[idx[:len(x)]]
        y_ = all[idx[len(x):]]
        diffs.append(statistic(x_, y_))
    # 원본 표본보다 작거나 큰 경우의 p-값을 계산합니다.
    less_pvalue = np.sum(diffs < obs_diff) / 1000
    greater_pvalue = np.sum(diffs > obs_diff) / 1000
    # 둘 중 작은 p-값을 선택해 2를 곱하여 최종 p-값을 반환합니다.
    return obs_diff, np.minimum(less_pvalue, greater_pvalue) * 2

In [ ]:
permutation_test(python_books['대출건수'], cplus_books['대출건수'])

## 도서 대출건수 평균 비교하기(2): 파이썬 vs 자바스크립트

In [ ]:
java_books_indx = ns_book7['주제분류번호'].str.startswith('00') & \
                  ns_book7['도서명'].str.contains('자바스크립트')
java_books = ns_book7[java_books_indx]
java_books.head()

In [ ]:
print(len(java_books), np.mean(java_books['대출건수']))

In [ ]:
permutation_test(python_books['대출건수'], java_books['대출건수'])